In [ ]:
# ══════════════════════════════════════════════════════════════
#  SpatialGlue: RNA + Isoform PSI 整合
#  在 SpatialGlue 环境的 JupyterLab 中运行
#  内核：Python (SpatialGlue)
# ══════════════════════════════════════════════════════════════

In [ ]:
# Cell 1: 环境检测 & 导入包
# ─────────────────────────────────────────────────────────────
import torch
import sys, os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.sparse as sparse
import matplotlib.pyplot as plt
from scipy.sparse import issparse
 
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
 
plt.rcParams.update({"pdf.fonttype": 42, "font.sans-serif": "Arial", "font.size": 10})
sc.set_figure_params(dpi=100, dpi_save=300, vector_friendly=True)
 
# R 路径（mclust 需要）
os.environ["R_HOME"] = "/histor/zhao/lupeng/micromamba/envs/SpatialGlue/lib/R"
 
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

# 路径配置
ISOFORM_DIR = "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260427_AS_analysis/Fig2_isoform"
ADATA_PATH  = "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260427_AS_analysis/adatas/G21-4_anno.h5ad"
OUT_DIR     = "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260427_AS_analysis/Fig2_isoform/SpatialGlue"
FIG_DIR     = os.path.join(OUT_DIR, "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
 
print("Cell 1 完成")

In [ ]:
# Cell 1: 环境检测 & 导入包
# ─────────────────────────────────────────────────────────────
import torch
import sys, os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.sparse as sparse
import matplotlib.pyplot as plt
from scipy.sparse import issparse
 
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
 
plt.rcParams.update({"pdf.fonttype": 42, "font.sans-serif": "Arial", "font.size": 10})
sc.set_figure_params(dpi=100, dpi_save=300, vector_friendly=True)
 
# R 路径（mclust 需要）
os.environ["R_HOME"] = "/histor/zhao/lupeng/micromamba/envs/SpatialGlue/lib/R"
 
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

# 路径配置
ISOFORM_DIR = "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260427_AS_analysis/Fig2_isoform"
ADATA_PATH  = "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260427_AS_analysis/adatas/G21-4_anno.h5ad"
OUT_DIR     = "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260427_AS_analysis/Fig2_isoform/SpatialGlue"
FIG_DIR     = os.path.join(OUT_DIR, "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
 
print("Cell 1 完成")

In [ ]:
# Cell 2: 加载数据
# ─────────────────────────────────────────────────────────────
def to_arr(adata):
    return adata.X.toarray() if issparse(adata.X) else np.array(adata.X)
 
# 1. RNA adata
print("加载 RNA adata...")
rna = sc.read(ADATA_PATH)
print(f"  RNA: {rna.shape}")
 
# 2. inclusion / skipping adata（归一化后的）
print("加载 isoform adata...")
inc_raw = ad.read_h5ad(os.path.join(ISOFORM_DIR, "inclusion_norm.h5ad"))
ski_raw = ad.read_h5ad(os.path.join(ISOFORM_DIR, "skipping_norm.h5ad"))
print(f"  inclusion: {inc_raw.shape}")
print(f"  skipping:  {ski_raw.shape}")
 
# spots 对齐
common_spots = inc_raw.obs_names.intersection(rna.obs_names)
inc_raw = inc_raw[common_spots].copy()
ski_raw = ski_raw[common_spots].copy()
rna_sub = rna[common_spots].copy()
print(f"  对齐后 spots: {len(common_spots)}")
 
# 差异事件筛选（FDR + dPSI）
FDR_THRESH  = 0.05
DPSI_THRESH = 0.05
sig_mask_inc = (
    (inc_raw.var["FDR"] <= FDR_THRESH) &
    (inc_raw.var["dPSI"].abs() > DPSI_THRESH) &
    (inc_raw.var["FDR"].notna())
)
sig_mask_ski = (
    (ski_raw.var["FDR"] <= FDR_THRESH) &
    (ski_raw.var["dPSI"].abs() > DPSI_THRESH) &
    (ski_raw.var["FDR"].notna())
)
inc_sig = inc_raw[:, sig_mask_inc].copy()
ski_sig = ski_raw[:, sig_mask_ski].copy()
print(f"  差异 inclusion 事件: {inc_sig.shape[1]}")
print(f"  差异 skipping  事件: {ski_sig.shape[1]}")
 

In [ ]:
# Cell 3: 计算 PSI 矩阵
# ─────────────────────────────────────────────────────────────
common_events = inc_sig.var_names.intersection(ski_sig.var_names)
print(f"共有事件数（用于 PSI）: {len(common_events)}")
 
inc_psi_arr = to_arr(inc_sig[:, common_events])
ski_psi_arr = to_arr(ski_sig[:, common_events])
total_arr   = inc_psi_arr + ski_psi_arr
 
with np.errstate(invalid="ignore", divide="ignore"):
    psi_mat = np.where(total_arr > 0, inc_psi_arr / total_arr, np.nan)
 
# NaN 填 0
psi_filled = np.where(np.isnan(psi_mat), 0.0, psi_mat)
print(f"PSI 矩阵: {psi_filled.shape}")
print(f"非零比例: {(psi_filled > 0).mean():.2%}")
 
print("Cell 3 完成")

In [ ]:
adata_omics1

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 4: 构建 SpatialGlue 输入
# ─────────────────────────────────────────────────────────────
from SpatialGlue.preprocess import fix_seed, pca, construct_neighbor_graph
fix_seed(2022)

# ── omics1：RNA ──
adata_omics1 = rna_sub.copy()
adata_omics1.var_names_make_unique()

# 直接用已有的 highly_variable 列，不重新计算 HVG
print(f"已有 HVG 数量: {adata_omics1.var['highly_variable'].sum()}")
sc.pp.scale(adata_omics1)
adata_omics1_high = adata_omics1[:, adata_omics1.var["highly_variable"]]
adata_omics1.obsm["feat"] = pca(adata_omics1_high, n_comps=50)
print(f"omics1 (RNA) feat shape: {adata_omics1.obsm['feat'].shape}")

# ── omics2：PSI isoform 矩阵 ──
adata_omics2 = ad.AnnData(
    X   = psi_filled,
    obs = inc_sig.obs.copy(),
    var = pd.DataFrame(index=list(common_events))
)
adata_omics2.obsm["spatial"] = rna_sub.obsm["spatial"]
adata_omics2.uns["spatial"]  = rna_sub.uns["spatial"]
adata_omics2.var_names_make_unique()

# 过滤低覆盖事件（至少 1% 的 spot 有值）
nonzero_ratio = (adata_omics2.X > 0).sum(axis=0) / adata_omics2.n_obs
adata_omics2  = adata_omics2[:, nonzero_ratio >= 0.01].copy()
print(f"过滤后 PSI isoform 数: {adata_omics2.shape[1]}")

# PSI 在 0-1 之间，直接 scale + PCA
sc.pp.highly_variable_genes(
    adata_omics2, flavor="seurat_v3",
    n_top_genes=min(3000, adata_omics2.shape[1])
)
sc.pp.scale(adata_omics2)
adata_omics2_high = adata_omics2[:, adata_omics2.var["highly_variable"]]
n_comps = min(50, adata_omics2_high.shape[1] - 1)
adata_omics2.obsm["feat"] = pca(adata_omics2_high, n_comps=n_comps)
print(f"omics2 (PSI) feat shape: {adata_omics2.obsm['feat'].shape}")

# obs_names 确认一致
assert (adata_omics1.obs_names == adata_omics2.obs_names).all(), \
    "obs_names 不一致，请检查"
print("obs_names 对齐: OK")
print("Cell 4 完成")

In [ ]:
# Cell 5: SpatialGlue 训练
# ─────────────────────────────────────────────────────────────
from SpatialGlue.SpatialGlue_pyG import Train_SpatialGlue
from SpatialGlue.utils import clustering
 
data_type = "10x"   # RNA + isoform PSI，类比 RNA + ADT
 
data   = construct_neighbor_graph(adata_omics1, adata_omics2, datatype=data_type)
model  = Train_SpatialGlue(data, datatype=data_type, device=device)
output = model.train()
 
print("训练完成")

In [ ]:
# Cell 6: 保存结果 & 聚类
# ─────────────────────────────────────────────────────────────
adata_sg = adata_omics1.copy()
adata_sg.obsm["emb_latent_omics1"] = output["emb_latent_omics1"].copy()
adata_sg.obsm["emb_latent_omics2"] = output["emb_latent_omics2"].copy()
adata_sg.obsm["SpatialGlue"]       = output["SpatialGlue"].copy()
adata_sg.obsm["alpha"]             = output["alpha"]
adata_sg.obsm["alpha_omics1"]      = output["alpha_omics1"]
adata_sg.obsm["alpha_omics2"]      = output["alpha_omics2"]
adata_sg.obsm["spatial"]           = rna_sub.obsm["spatial"]
adata_sg.uns["spatial"]            = rna_sub.uns["spatial"]
adata_sg.obs["AS_cluster"]         = rna_sub.obs["AS_cluster"].values

# 聚类（n_clusters 对应脑区数）
N_CLUSTERS = 10
clustering(
    adata_sg, key="SpatialGlue", add_key="SpatialGlue",
    n_clusters=N_CLUSTERS, method="leiden", use_pca=True
)
print(f"聚类完成，共 {N_CLUSTERS} 个簇")


In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 7: 可视化（修复版）
# ─────────────────────────────────────────────────────────────
sc.pp.neighbors(adata_sg, use_rep="SpatialGlue", n_neighbors=30)
sc.tl.umap(adata_sg)

# ── UMAP + 空间图 ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sc.pl.umap(adata_sg, color="SpatialGlue",
           ax=axes[0], show=False, title="SpatialGlue UMAP")
sc.pl.umap(adata_sg, color="AS_cluster",
           ax=axes[1], show=False, title="AS_cluster (reference)")
sc.pl.spatial(adata_sg, color="SpatialGlue",
              img_key=None, spot_size=26,
              ax=axes[2], show=False, title="SpatialGlue spatial")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "SpatialGlue_RNA_PSI.pdf"), dpi=300, bbox_inches="tight")
plt.show(); plt.close()

# ── alpha 图：先把 obsm 转存到 obs ──
# alpha_omics1/2 是每个 spot 的标量权重，取第一列
adata_sg.obs["alpha_RNA"]     = adata_sg.obsm["alpha_omics1"][:, 0] \
                                  if adata_sg.obsm["alpha_omics1"].ndim > 1 \
                                  else adata_sg.obsm["alpha_omics1"]
adata_sg.obs["alpha_isoform"] = adata_sg.obsm["alpha_omics2"][:, 0] \
                                  if adata_sg.obsm["alpha_omics2"].ndim > 1 \
                                  else adata_sg.obsm["alpha_omics2"]

print(f"alpha_RNA 范围: {adata_sg.obs['alpha_RNA'].min():.3f} ~ {adata_sg.obs['alpha_RNA'].max():.3f}")
print(f"alpha_isoform 范围: {adata_sg.obs['alpha_isoform'].min():.3f} ~ {adata_sg.obs['alpha_isoform'].max():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc.pl.spatial(
    adata_sg, color="alpha_RNA",
    img_key=None, spot_size=26,
    cmap="Reds", ax=axes[0], show=False,
    title="Alpha: RNA weight"
)
sc.pl.spatial(
    adata_sg, color="alpha_isoform",
    img_key=None, spot_size=26,
    cmap="Blues", ax=axes[1], show=False,
    title="Alpha: PSI isoform weight"
)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "SpatialGlue_alpha.pdf"), dpi=300, bbox_inches="tight")
plt.show(); plt.close()

print("Cell 7 完成")

In [ ]:
adata_sg

In [ ]:
sc.pp.neighbors(adata_sg, n_neighbors=50, use_rep="SpatialGlue_pca")
sc.tl.leiden(adata_sg, random_state=0, resolution=0.78, key_added="SpatialGlue_15")

In [ ]:
sc.pl.spatial(adata_sg, color="SpatialGlue_15",
              img_key=None, spot_size=26)

# 新psi矩阵

In [ ]:
import os
import torch
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
# Environment configuration. SpatialGlue pacakge can be implemented with either CPU or GPU. GPU acceleration is highly recommend for imporoved efficiency.
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# the location of R, which is necessary for mclust algorithm. Please replace the path below with local R installation path
REAL_R_HOME = "/histor/zhao/lupeng/micromamba/envs/SpatialGlue/lib/R"
os.environ['R_HOME'] = REAL_R_HOME

In [ ]:
# read data
ADATA_PATH  = "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260427_AS_analysis/adatas/G21-4_anno.h5ad"
PSI_PATH = "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260427_AS_analysis/Fig2_isoform/psi_add1_adata.h5ad"
adata_omics1 = sc.read_h5ad(ADATA_PATH)
adata_omics2 = sc.read_h5ad(PSI_PATH)

adata_omics1.var_names_make_unique()
adata_omics2.var_names_make_unique()
print(adata_omics1)
print(adata_omics2)

In [ ]:
# Specify data type
data_type = '10x'

# Fix random seed
from SpatialGlue.preprocess import fix_seed
random_seed = 2022
fix_seed(random_seed)

In [ ]:
sc.pp.scale(adata_omics1)
adata_omics1_high = adata_omics1[:, adata_omics1.var["highly_variable"]]
adata_omics1.obsm["feat"] = pca(adata_omics1_high, n_comps=50)

In [ ]:
sc.pp.filter_genes(adata_omics2, min_cells=10)
adata_omics2

In [ ]:
X = adata_omics2.X.toarray() if issparse(adata_omics2.X) else np.array(adata_omics2.X)
print(f"shape: {adata_omics2.shape}")
print(f"非零比例: {(X != 0).mean():.2%}")
print(f"值域: {X.min():.3f} ~ {X.max():.3f}")
print(f"每个spot平均有效事件数: {(X != 0).sum(axis=1).mean():.1f}")
print(f"每个事件平均覆盖spot数: {(X != 0).sum(axis=0).mean():.1f}")

## 插曲

In [ ]:
common = adata_omics1.obs_names.intersection(adata_omics2.obs_names)
adata_omics1 = adata_omics1[common].copy()
adata_omics2 = adata_omics2[common].copy()

In [ ]:
# 2. 不用highly_variable_genes，直接用pseudo-bulk方差筛选
#    重新计算event_var
from scipy.sparse import issparse
import pandas as pd

inc_raw = ad.read_h5ad("/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260427_AS_analysis/Fig2_isoform/inclusion_norm.h5ad")
ski_raw = ad.read_h5ad("/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260427_AS_analysis/Fig2_isoform/skipping_norm.h5ad")
inc_raw = inc_raw[common].copy()
ski_raw = ski_raw[common].copy()

# 只取adata_omics2现有的事件
common_psi = adata_omics2.var_names
inc_c = np.asarray(inc_raw[:, common_psi].layers["counts"])
ski_c = np.asarray(ski_raw[:, common_psi].layers["counts"])

clusters = adata_omics1.obs["AS_cluster"]
cluster_ids = sorted(clusters.unique())

psi_pb = {}
for cl in cluster_ids:
    idx = np.where(clusters == cl)[0]
    inc_sum = inc_c[idx].sum(axis=0)
    ski_sum = ski_c[idx].sum(axis=0)
    total = inc_sum + ski_sum
    with np.errstate(invalid="ignore", divide="ignore"):
        psi_pb[cl] = np.where(total > 0, inc_sum / total, np.nan)

psi_pb_df = pd.DataFrame(psi_pb, index=common_psi).T
event_var  = psi_pb_df.var(axis=0, skipna=True)
event_cov  = psi_pb_df.notna().mean(axis=0)

keep = (event_var >= 0.01) & (event_cov >= 0.5)
print(f"pseudo-bulk方差筛选后: {keep.sum()} 个事件")

adata_omics2 = adata_omics2[:, keep[keep].index].copy()
print(f"最终omics2: {adata_omics2.shape}")


In [ ]:
# 3. 标记highly_variable（SpatialGlue需要这个字段）
adata_omics2.var['highly_variable'] = True

print(f"\nomics1: {adata_omics1.shape}")
print(f"omics2: {adata_omics2.shape}")

In [ ]:
adata_omics2_high = adata_omics2[:, adata_omics2.var['highly_variable']]
n_comps2 = min(30, adata_omics2_high.shape[1]-1, adata_omics2_high.shape[0]-1)
adata_omics2.obsm['feat'] = pca(adata_omics2_high, n_comps=n_comps2)
print(f"omics2 feat: {adata_omics2.obsm['feat'].shape}")

In [ ]:
from SpatialGlue.preprocess import construct_neighbor_graph
from SpatialGlue.SpatialGlue_pyG import Train_SpatialGlue
from SpatialGlue.utils import clustering
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

data_type = "10x"
data   = construct_neighbor_graph(adata_omics1, adata_omics2, datatype=data_type)
model  = Train_SpatialGlue(data, datatype=data_type, device=device)
output = model.train()
print("训练完成")

## buzhidao

In [ ]:
# PSI

sc.pp.highly_variable_genes(
    adata_omics2, flavor="seurat_v3",
    n_top_genes=min(7800, adata_omics2.shape[1])
)
sc.pp.scale(adata_omics2)
adata_omics2_high = adata_omics2[:, adata_omics2.var["highly_variable"]]
n_comps = min(30, adata_omics2_high.shape[1] - 1)
adata_omics2.obsm["feat"] = pca(adata_omics2_high, n_comps=n_comps)

In [ ]:
# obs_names 确认一致
assert (adata_omics1.obs_names == adata_omics2.obs_names).all(), \
    "obs_names 不一致，请检查"
print("obs_names 对齐: OK")
print("Cell 4 完成")

In [ ]:
# Cell 5: SpatialGlue 训练
# ─────────────────────────────────────────────────────────────
from SpatialGlue.SpatialGlue_pyG import Train_SpatialGlue
from SpatialGlue.utils import clustering
 
data_type = "10x"   # RNA + isoform PSI，类比 RNA + ADT
 
data   = construct_neighbor_graph(adata_omics1, adata_omics2, datatype=data_type)
model  = Train_SpatialGlue(data, datatype=data_type, device=device)
output = model.train()
 
print("训练完成")

# 可视化

In [ ]:
# Cell 6: 保存结果 & 聚类
# ─────────────────────────────────────────────────────────────
adata_sg = adata_omics1.copy()
adata_sg.obsm["emb_latent_omics1"] = output["emb_latent_omics1"].copy()
adata_sg.obsm["emb_latent_omics2"] = output["emb_latent_omics2"].copy()
adata_sg.obsm["SpatialGlue"]       = output["SpatialGlue"].copy()
adata_sg.obsm["alpha"]             = output["alpha"]
adata_sg.obsm["alpha_omics1"]      = output["alpha_omics1"]
adata_sg.obsm["alpha_omics2"]      = output["alpha_omics2"]
adata_sg.obsm["spatial"]           = rna_sub.obsm["spatial"]
adata_sg.uns["spatial"]            = rna_sub.uns["spatial"]
adata_sg.obs["AS_cluster"]         = rna_sub.obs["AS_cluster"].values

# 聚类（n_clusters 对应脑区数）
N_CLUSTERS = 28
clustering(
    adata_sg, key="SpatialGlue", add_key="SpatialGlue",
    n_clusters=N_CLUSTERS, method="leiden", use_pca=True
)
print(f"聚类完成，共 {N_CLUSTERS} 个簇")


In [ ]:
color_list=['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf',
  '#a6cee3','#fdbf6f','#b2df8a','#fb9a99','#cab2d6','#f3cb29','#a30059','#22a884','#ff34ff','#809693','#ffdbe5','#023fa5','#9cded6']

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 7: 可视化（修复版）
# ─────────────────────────────────────────────────────────────
sc.pp.neighbors(adata_sg, use_rep="SpatialGlue", n_neighbors=30)
sc.tl.umap(adata_sg)

# ── UMAP + 空间图 ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sc.pl.umap(adata_sg, color="SpatialGlue",palette=color_list,
           ax=axes[0], show=False, title="SpatialGlue UMAP")
sc.pl.umap(adata_sg, color="AS_cluster",
           ax=axes[1], show=False, title="AS_cluster (reference)")
sc.pl.spatial(adata_sg, color="SpatialGlue",palette=color_list,
              img_key=None, spot_size=26,
              ax=axes[2], show=False, title="SpatialGlue spatial")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "SpatialGlue_RNA_PSI.pdf"), dpi=300, bbox_inches="tight")
plt.show(); plt.close()

# ── alpha 图：先把 obsm 转存到 obs ──
# alpha_omics1/2 是每个 spot 的标量权重，取第一列
adata_sg.obs["alpha_RNA"]     = adata_sg.obsm["alpha_omics1"][:, 0] \
                                  if adata_sg.obsm["alpha_omics1"].ndim > 1 \
                                  else adata_sg.obsm["alpha_omics1"]
adata_sg.obs["alpha_isoform"] = adata_sg.obsm["alpha_omics2"][:, 0] \
                                  if adata_sg.obsm["alpha_omics2"].ndim > 1 \
                                  else adata_sg.obsm["alpha_omics2"]

print(f"alpha_RNA 范围: {adata_sg.obs['alpha_RNA'].min():.3f} ~ {adata_sg.obs['alpha_RNA'].max():.3f}")
print(f"alpha_isoform 范围: {adata_sg.obs['alpha_isoform'].min():.3f} ~ {adata_sg.obs['alpha_isoform'].max():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc.pl.spatial(
    adata_sg, color="alpha_RNA",
    img_key=None, spot_size=26,
    cmap="Reds", ax=axes[0], show=False,
    title="Alpha: RNA weight"
)
sc.pl.spatial(
    adata_sg, color="alpha_isoform",
    img_key=None, spot_size=26,
    cmap="Blues", ax=axes[1], show=False,
    title="Alpha: PSI isoform weight"
)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "SpatialGlue_alpha.pdf"), dpi=300, bbox_inches="tight")
plt.show(); plt.close()

print("Cell 7 完成")

In [ ]:
# 快速诊断
print(f"alpha_RNA    mean={adata_sg.obs['alpha_RNA'].mean():.4f}  std={adata_sg.obs['alpha_RNA'].std():.4f}")
print(f"alpha_isoform mean={adata_sg.obs['alpha_isoform'].mean():.4f}  std={adata_sg.obs['alpha_isoform'].std():.4f}")

# 按cluster看
print("\nMean alpha per AS_cluster:")
print(adata_sg.obs.groupby("AS_cluster")[["alpha_RNA","alpha_isoform"]].mean().round(4))

In [ ]:
# ══════════════════════════════════════════
# Cell: 权重贡献完整可视化
# ══════════════════════════════════════════
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc

fig = plt.figure(figsize=(20, 12))

# ── 1. 空间图：RNA权重 ──
ax1 = fig.add_subplot(2, 3, 1)
sc.pl.spatial(adata_sg, color="alpha_RNA",
              cmap="RdBu_r", vmin=0.2, vmax=0.8,
              img_key=None, spot_size=26,
              title="RNA weight (alpha)", ax=ax1, show=False)

# ── 2. 空间图：PSI权重 ──
ax2 = fig.add_subplot(2, 3, 2)
sc.pl.spatial(adata_sg, color="alpha_isoform",
              cmap="RdBu_r", vmin=0.2, vmax=0.8,
              img_key=None, spot_size=26,
              title="AS (PSI) weight (alpha)", ax=ax2, show=False)

# ── 3. 空间图：PSI - RNA差值（正值=AS主导，负值=RNA主导）──
adata_sg.obs["alpha_diff"] = adata_sg.obs["alpha_isoform"] - adata_sg.obs["alpha_RNA"]
ax3 = fig.add_subplot(2, 3, 3)
sc.pl.spatial(adata_sg, color="alpha_diff",
              cmap="RdBu_r", vmin=-0.3, vmax=0.3,
              img_key=None, spot_size=26,
              title="AS - RNA weight (positive = AS dominant)",
              ax=ax3, show=False)

# ── 4. 按cluster的权重柱状图 ──
ax4 = fig.add_subplot(2, 1, 2)
cluster_alpha = adata_sg.obs.groupby("AS_cluster")[["alpha_RNA","alpha_isoform"]].mean()
cluster_alpha = cluster_alpha.sort_values("alpha_isoform", ascending=False)

x = np.arange(len(cluster_alpha))
width = 0.35
bars1 = ax4.bar(x - width/2, cluster_alpha["alpha_RNA"],
                width, label="RNA", color="steelblue", alpha=0.8)
bars2 = ax4.bar(x + width/2, cluster_alpha["alpha_isoform"],
                width, label="AS (PSI)", color="tomato", alpha=0.8)

ax4.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="Equal weight (0.5)")
ax4.set_xticks(x)
ax4.set_xticklabels(cluster_alpha.index, rotation=45, ha="right")
ax4.set_ylabel("Mean alpha weight")
ax4.set_title("Modality contribution per brain region")
ax4.set_ylim(0.4, 0.6)
ax4.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "spatialglue_alpha_contribution.pdf"),
            bbox_inches="tight")
plt.show()

# ── 5. 统计检验：哪些cluster的AS权重显著高于RNA ──
from scipy import stats
print("\nPaired t-test per cluster (PSI vs RNA alpha):")
print(f"{'Cluster':<12} {'PSI_mean':>10} {'RNA_mean':>10} {'diff':>8} {'p-value':>10} {'sig':>5}")
print("-" * 60)
for cl in cluster_alpha.index:
    mask = adata_sg.obs["AS_cluster"] == cl
    psi_vals = adata_sg.obs.loc[mask, "alpha_isoform"]
    rna_vals = adata_sg.obs.loc[mask, "alpha_RNA"]
    t, p = stats.ttest_rel(psi_vals, rna_vals)
    diff = psi_vals.mean() - rna_vals.mean()
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    print(f"{cl:<12} {psi_vals.mean():>10.4f} {rna_vals.mean():>10.4f} {diff:>8.4f} {p:>10.4f} {sig:>5}")

In [ ]:
# 只用RNA跑一遍SpatialGlue（把omics2换成随机噪声）
import numpy as np

# 构造一个随机噪声的omics2作为对照
adata_omics2_random = adata_omics2.copy()
np.random.seed(42)
adata_omics2_random.X = np.random.rand(*adata_omics2.X.shape) * 2  # 随机[0,2]
adata_omics2_random.obsm['feat'] = np.random.rand(
    adata_omics2.shape[0], 
    adata_omics2.obsm['feat'].shape[1]
)

# 用随机omics2跑SpatialGlue
data_random = construct_neighbor_graph(adata_omics1, adata_omics2_random, datatype="10x")
model_random = Train_SpatialGlue(data_random, datatype="10x", device=device)
output_random = model_random.train()

alpha_random = output_random["alpha_omics2"]
alpha_random = np.array(alpha_random).flatten()
print(f"随机噪声 alpha_omics2: mean={alpha_random.mean():.4f}")

# 0718重新SG

In [ ]:
# Cell 1: 环境检测 & 导入包
# ─────────────────────────────────────────────────────────────
import torch
import sys, os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.sparse as sparse
import matplotlib.pyplot as plt
from scipy.sparse import issparse
from SpatialGlue.preprocess import fix_seed, pca, construct_neighbor_graph

In [ ]:
# Environment configuration. SpatialGlue pacakge can be implemented with either CPU or GPU. GPU acceleration is highly recommend for imporoved efficiency.
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# the location of R, which is necessary for mclust algorithm. Please replace the path below with local R installation path
REAL_R_HOME = "/histor/zhao/lupeng/micromamba/envs/SpatialGlue/lib/R"
os.environ['R_HOME'] = REAL_R_HOME

In [ ]:
# read data
ADATA_PATH  = "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260427_AS_analysis/adatas/G21-4_repaired_AS.h5ad"
PSI_PATH = "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260427_AS_analysis/Fig2_isoform/psi_add1_adata.h5ad"
adata_omics1 = sc.read_h5ad(ADATA_PATH)
adata_omics2 = sc.read_h5ad(PSI_PATH)

adata_omics1.var_names_make_unique()
adata_omics2.var_names_make_unique()
print(adata_omics1)
print(adata_omics2)
print("#########################")
print(adata_omics1.obs)
print("#########################")
print(adata_omics2.obs)


In [ ]:
adata_omics1.obs_names = adata_omics1.obs_names.astype(str) + "_G21-4"
adata_omics1.obs

In [ ]:
# Specify data type
data_type = '10x'

# Fix random seed
from SpatialGlue.preprocess import fix_seed
random_seed = 2022
fix_seed(random_seed)

In [ ]:
sc.pp.scale(adata_omics1)
adata_omics1_high = adata_omics1[:, adata_omics1.var["highly_variable"]]
adata_omics1.obsm["feat"] = pca(adata_omics1_high, n_comps=50)

In [ ]:
sc.pp.filter_genes(adata_omics2, min_cells=10)
adata_omics2

In [ ]:
X = adata_omics2.X.toarray() if issparse(adata_omics2.X) else np.array(adata_omics2.X)
print(f"shape: {adata_omics2.shape}")
print(f"非零比例: {(X != 0).mean():.2%}")
print(f"值域: {X.min():.3f} ~ {X.max():.3f}")
print(f"每个spot平均有效事件数: {(X != 0).sum(axis=1).mean():.1f}")
print(f"每个事件平均覆盖spot数: {(X != 0).sum(axis=0).mean():.1f}")

In [ ]:
common = adata_omics1.obs_names.intersection(adata_omics2.obs_names)
adata_omics1 = adata_omics1[common].copy()
adata_omics2 = adata_omics2[common].copy()

In [ ]:
adata_omics2.obsm['feat'] = pca(adata_omics2, n_comps=adata_omics2.n_vars-1)

In [ ]:
from SpatialGlue.preprocess import construct_neighbor_graph
data = construct_neighbor_graph(adata_omics1, adata_omics2, datatype=data_type)

In [ ]:
# define model
from SpatialGlue.SpatialGlue_pyG import Train_SpatialGlue
model = Train_SpatialGlue(data, datatype=data_type, device=device)

# train model
output = model.train()

In [ ]:
adata = adata_omics1.copy()
adata.obsm['emb_latent_omics1'] = output['emb_latent_omics1'].copy()
adata.obsm['emb_latent_omics2'] = output['emb_latent_omics2'].copy()
adata.obsm['SpatialGlue'] = output['SpatialGlue'].copy()
adata.obsm['alpha'] = output['alpha']
adata.obsm['alpha_omics1'] = output['alpha_omics1']
adata.obsm['alpha_omics2'] = output['alpha_omics2']

In [ ]:
# we set 'mclust' as clustering tool by default. Users can also select 'leiden' and 'louvain'
from SpatialGlue.utils import clustering
tool = 'leiden' # mclust, leiden, and louvain
clustering(adata, key='SpatialGlue', add_key='SpatialGlue', n_clusters=29, method=tool, use_pca=True)

In [ ]:
# visualization
import matplotlib.pyplot as plt
fig, ax_list = plt.subplots(1, 2, figsize=(14, 6))
sc.pp.neighbors(adata, use_rep='SpatialGlue', n_neighbors=10)
sc.tl.umap(adata)

sc.pl.umap(adata, color='SpatialGlue', ax=ax_list[0], title='SpatialGlue', s=25, show=False)
sc.pl.embedding(adata, basis='spatial', color='SpatialGlue', ax=ax_list[1], title='SpatialGlue', s=28, show=False)

plt.tight_layout(w_pad=0.3)
plt.show()

In [ ]:
color_list = [
    "#1F77B4", "#FF7F0E", "#2CA02C", "#D62728", "#9467BD", "#8C564B",
    "#E377C2", "#7F7F7F", "#BCBD22", "#17BECF",
    "#D1495B", "#5E60CE", "#FF99C8", "#A8DADC",
    "#A6CEE3", "#FDBF6F", "#B2DF8A", "#FB9A99", "#CAB2D6", "#F3CB29",
    "#A30059", "#22A884", "#FF34FF", "#809693", "#FFDBE5", "#023FA5",
    "#9CDED6",
    
    "#E76F51", "#457B9D", "#6A994E", "#C44536", "#7B2CBF", "#B56576",
    "#00B4D8", "#90BE6D", "#F9C74F", "#F9844A", "#577590", "#43AA8B",
]

In [ ]:
sc.pl.spatial(
    adata, color="SpatialGlue",palette=color_list,
    img_key=None, spot_size=28,)

In [ ]:
sc.tl.leiden(adata, restrict_to=["SpatialGlue", ["20"]], resolution=0.3, key_added='leiden1')

In [ ]:
sc.tl.leiden(adata, restrict_to=["leiden1", ["19"]], resolution=0.2, key_added='leiden2')


In [ ]:
sc.pl.spatial(
    adata, color="leiden1",palette=color_list,
    img_key=None, spot_size=28,)

In [ ]:
sc.pl.spatial(
    adata, color="leiden2",palette=color_list,
    img_key=None, spot_size=28,)

In [ ]:
sc.tl.leiden(adata, restrict_to=["leiden2", ["25"]], resolution=0.2, key_added='leiden3')


In [ ]:
sc.pl.spatial(
    adata, color="leiden3",palette=color_list,
    img_key=None, spot_size=28,)

In [ ]:
sc.pl.spatial(
    adata, color="leiden3",palette=color_list,groups="15",
    img_key=None, spot_size=28,)

In [ ]:
plt.rcParams['figure.figsize']=(6,6)
import seaborn as sns
sns.set(style="ticks", font_scale=1.2)

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

# ── 字体设置 ───────────────────────────────────────────
plt.rcParams["font.family"] = "Arial"
plt.rcParams["pdf.fonttype"] = 42   # PDF 中保留可编辑文字
plt.rcParams["ps.fonttype"] = 42

# ── 输出路径 ───────────────────────────────────────────
OUTDIR = Path("/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260427_AS_analysis/Fig2_isoform/SpatialGlue/figures0718")
OUTDIR.mkdir(exist_ok=True)

# ── Spatial 图 ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))

sc.pl.spatial(
    adata,
    color="leiden3",
    palette=color_list,
    img_key=None,
    spot_size=28,
    ax=ax,
    show=False,
)

fig.savefig(
    OUTDIR / "leiden3_spatial.pdf",
    bbox_inches="tight",
    transparent=True,
)
plt.show()

# ── UMAP 图 ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

sc.pl.umap(
    adata,
    color="leiden3",
    palette=color_list,
    #legend_loc="on data",      # 想在右侧显示图例可改为 "right margin"
    frameon=False,
    ax=ax,
    show=False,
)

fig.savefig(
    OUTDIR / "leiden3_umap.pdf",
    bbox_inches="tight",
    transparent=True,
)
plt.show()
plt.close()

In [ ]:
adata.write("/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260427_AS_analysis/Fig2_spatialGlue/adatas/G21-4_SG-gene-psi.h5ad")

In [ ]:
adata

In [ ]:
adata.obs['leiden3'].unique().tolist()

In [ ]:
adata.uns['leiden3_colors']

In [ ]:
sc.pl.spatial(adata,color="leiden3",groups="3",img_key=None,spot_size=25)

In [ ]:
sc.pl.spatial(adata,color="leiden3",groups=["3","10"],img_key=None,spot_size=25)

## 对SG改色

In [ ]:
adata=sc.read("/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260427_AS_analysis/Fig2_spatialGlue/adatas/G21-4_SG-gene-psi.h5ad")

In [ ]:
# 修正后的 Leiden cluster → G21-4 解卷积主导细胞类型
leiden3_to_celltype = {
    "0":    "Inh_4",
    "1":    "Ext_Thal_1",
    "2":    "Ext_Thal_1",
    "3":    "Astro_WM",
    "4":    "Ext_L5_2",
    "5":    "Inh_Meis2_3",
    "6":    "Ext_Pir",
    "7":    "Ext_ClauPyr",
    "8":    "Oligo_1",
    "9":    "Inh_1",
    "10":   "Ext_L56",
    "11":   "Ext_L23",
    "12":   "Ext_Thal_2",
    "13":   "Ext_L23",
    "14":   "Oligo_1",
    "15":   "Ext_Hpc_DG1",
    "16":   "Ext_L25",
    "17":   "Ext_Hpc_CA3",
    "18":   "Ext_Hpc_CA1",
    "19,0": "Ext_Hpc_DG1",
    "19,1": "Oligo_2",
    "20,0": "Ext_Pir",
    "20,1": "Ext_Thal_1",
    "20,2": "Oligo_1",
    "21":   "Ext_Pir",
    "22":   "Ext_Amy_2",
    "23":   "Ext_Thal_1",
    "24":   "Ext_Hpc_CA3",
    "25,0": "Inh_6",
    "25,1": "Astro_WM",
    "26":   "Oligo_1",
    "27":   "Astro_WM",
    "28":   "Ext_Hpc_CA3",
}

celltype_colors = {
    "Ext_Thal_1":   "#cac43d",
    "Oligo_1":      "#10976f",
    "Ext_L23":      "#1681ec",
    "Ext_L25":      "#a81e2b",
    "Ext_Hpc_DG1":  "#009e73",
    "Oligo_2":      "#67c1dd",
    "Ext_L56":      "#5a14a5",
    "Ext_Hpc_CA1":  "#f0e442",
    "Ext_Thal_2":   "#b71e75",
    "Ext_Pir":      "#74c476",
    "Ext_Amy_2":    "#8c510a",
    "Ext_L5_2":     "#e6550d",
    "Inh_4":        "#ffbe2c",
    "Inh_1":        "#865123",
    "Ext_Hpc_CA3":  "#56b4e9",
    "Inh_Meis2_3":  "#6baed6",
    "Astro_WM":     "#e7969c",
    "Ext_ClauPyr":  "#756bb1",
    "Inh_6":        "#636363",
}

leiden3_palette = {
    cluster: celltype_colors[celltype]
    for cluster, celltype in leiden3_to_celltype.items()
}

# 覆盖原来的颜色
adata.obs["leiden3"] = (
    adata.obs["leiden3"]
    .astype(str)
    .astype("category")
)

category_order = adata.obs["leiden3"].cat.categories.tolist()

adata.uns["leiden3_colors"] = [
    leiden3_palette[cluster]
    for cluster in category_order
]

In [ ]:
sc.pl.spatial(
    adata,
    color="leiden3",
    palette=leiden3_palette,
    img_key=None,
    spot_size=28,
    frameon=False,
    # ax=ax,
    # show=False,
)

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

# ── 字体设置 ───────────────────────────────────────────
plt.rcParams["font.family"] = "Arial"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

# ── 按 leiden3 category 顺序生成颜色 ────────────────────
adata.obs["leiden3"] = adata.obs["leiden3"].astype("category")

color_list = [
    leiden3_color_map[str(cluster)]
    for cluster in adata.obs["leiden3"].cat.categories
]
adata.uns["leiden3_colors"] = color_list

# ── 输出路径 ───────────────────────────────────────────
OUTDIR = Path(
    "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/"
    "20260427_AS_analysis/Fig2_isoform/SpatialGlue/figures0718"
)
OUTDIR.mkdir(parents=True, exist_ok=True)

# ── Spatial 图：无 hires，黑色背景 ──────────────────────
with plt.style.context("dark_background"):
    fig, ax = plt.subplots(figsize=(6, 6), facecolor="black")
    ax.set_facecolor("black")

    sc.pl.spatial(
        adata,
        color="leiden3",
        palette=leiden3_palette,
        img_key=None,      # 不使用 hires/组织学图片
        spot_size=28,
        ax=ax,
        show=False,
    )

    fig.savefig(
        OUTDIR / "leiden3_spatial_black.pdf",
        bbox_inches="tight",
        transparent=False,
        facecolor="black",
    )

    plt.show()
    plt.close(fig)

# ── UMAP 图：黑色背景 ──────────────────────────────────
with plt.style.context("dark_background"):
    fig, ax = plt.subplots(figsize=(7, 6), facecolor="black")
    ax.set_facecolor("black")

    sc.pl.umap(
        adata,
        color="leiden3",
        palette=leiden3_palette,
        frameon=False,
        ax=ax,
        show=False,
    )

    fig.savefig(
        OUTDIR / "leiden3_umap_black.pdf",
        bbox_inches="tight",
        transparent=False,
        facecolor="black",
    )

    plt.show()
    plt.close(fig)

## 手动改色

In [ ]:
sc.pl.spatial(adata,color="leiden3",groups=["10","11"],img_key=None,spot_size=25)

In [ ]:
deconv_colors_G214 = {
    "Ext_Thal_1":     "#cac43d",  # 黄绿色
    "Oligo_1":        "#10976f",  # 深青绿
    "Ext_L23":        "#1681ec",  # 蓝色
    "Ext_L25":        "#a81e2b",  # 暗红色
    "Ext_Hpc_DG1":    "#009e73",  # 绿色
    "Oligo_2":        "#67c1dd",  # 浅蓝色
    "Ext_L56":        "#5a14a5",  # 紫色
    "Ext_Hpc_CA1":    "#f0e442",  # 明黄色
    "Ext_Thal_2":     "#b71e75",  # 洋红色
    "Ext_Pir":        "#74c476",  # 浅绿色
    "Ext_Amy_2":      "#8c510a",  # 棕色
    "Ext_L5_1":       "#fd8d3c",  # 橙色
    "Inh_4":          "#ffbe2c",  # 金黄色
    "Inh_1":          "#865123",  # 深棕色
    "Ext_Hpc_CA3":    "#56b4e9",  # 天蓝色
    "Micro":          "#9e9ac8",  # 淡紫色
    "Inh_Meis2_3":    "#6baed6",  # 浅蓝
    "Ext_L6":         "#31a354",  # 草绿色
    "Ext_L5_2":       "#e6550d",  # 橙红色
    "Astro_HYPO":     "#8ca252",  # 橄榄绿色
    "Ext_L6B":        "#3182bd",  # 中蓝色
    "Endo":           "#ec7c7f",  # 珊瑚粉
    "Astro_WM":       "#e7969c",  # 浅粉色
    "Inh_Pvalb":      "#ce6dbd",  # 粉紫色
    "Ext_ClauPyr":    "#756bb1",  # 蓝紫色
    "Nb_2":           "#2ca02c",  # 绿色
    "Inh_5":          "#7b4173",  # 深紫色
    "Astro_THAL_lat": "#bd9e39",  # 土黄色
    "Inh_6":          "#636363",  # 深灰色
    "Inh_Meis2_4":    "#9c9ede",  # 淡蓝紫
    "Ext_Med":        "#a55194",  # 紫红色

    # notebook 中没有单独指定，所以使用默认补充颜色
    "Inh_Sst":        "#aaaaaa",  # 灰色
}

In [ ]:
leiden3_palette = {
    "0":    deconv_colors_G214["Ext_L56"],
    "1":    deconv_colors_G214["Ext_L5_1"],
    "2":    deconv_colors_G214["Oligo_1"],
    "3":    deconv_colors_G214["Ext_Pir"],
    "4":    deconv_colors_G214["Ext_Thal_2"],
    "5":    deconv_colors_G214["Inh_4"],
    "6":    deconv_colors_G214["Ext_L25"],
    "7":    deconv_colors_G214["Ext_ClauPyr"],
    "8":    deconv_colors_G214["Astro_THAL_lat"],
    "9":    deconv_colors_G214["Inh_1"],
    "10":   deconv_colors_G214["Ext_L6B"],
    "11":   deconv_colors_G214["Inh_Meis2_3"],
    "12":   deconv_colors_G214["Oligo_1"],
    "13":   deconv_colors_G214["Micro"],
    "14":   deconv_colors_G214["Inh_Pvalb"],
    "15":   deconv_colors_G214["Inh_6"],#"#000000",
    "16":   deconv_colors_G214["Ext_L25"],
    "17":   deconv_colors_G214["Ext_L5_2"],
    "18":   deconv_colors_G214["Ext_Amy_2"],
    "19,0": deconv_colors_G214["Inh_4"],
    "19,1": deconv_colors_G214["Ext_Hpc_CA1"],
    "20,0": deconv_colors_G214["Inh_6"],
    "20,1": deconv_colors_G214["Ext_Hpc_CA3"],
    "20,2": deconv_colors_G214["Ext_Hpc_CA3"],
    "21":   deconv_colors_G214["Ext_L6"],
    "22":   deconv_colors_G214["Astro_HYPO"],
    "23":   deconv_colors_G214["Ext_Med"],
    "24":   deconv_colors_G214["Inh_5"],
    "25,0": deconv_colors_G214["Ext_Hpc_DG1"],
    "25,1": deconv_colors_G214["Inh_Pvalb"],
    "26":   deconv_colors_G214["Astro_HYPO"],
    "27":   deconv_colors_G214["Inh_1"],
    "28":   deconv_colors_G214["Endo"],
}

In [ ]:
# 确保 leiden3 是分类变量
adata.obs["leiden3"] = adata.obs["leiden3"].astype(str).astype("category")

# 检查是否每个 cluster 都有颜色
categories = adata.obs["leiden3"].cat.categories.tolist()
missing = set(categories) - set(leiden3_palette)

if missing:
    raise ValueError(f"这些 leiden3 cluster 没有设置颜色：{sorted(missing)}")

# 按 category 顺序写入
adata.uns["leiden3_colors"] = [
    leiden3_palette[cluster]
    for cluster in categories
]

# 检查结果
print(dict(zip(categories, adata.uns["leiden3_colors"])))

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path

# ── 字体设置 ───────────────────────────────────────────
plt.rcParams["font.family"] = "Arial"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

# 关键：关闭 Scanpy 自动栅格化，保证 PDF 中点尽量保持矢量可编辑
sc.settings._vector_friendly = False

# ── 按 leiden3 category 顺序生成颜色 ────────────────────
adata.obs["leiden3"] = adata.obs["leiden3"].astype("category")

color_list = [
    leiden3_color_map[str(cluster)]
    for cluster in adata.obs["leiden3"].cat.categories
]
adata.uns["leiden3_colors"] = color_list

# ── 输出路径 ───────────────────────────────────────────
OUTDIR = Path(
    "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/"
    "20260427_AS_analysis/Fig2_isoform/SpatialGlue/figures0718"
)
OUTDIR.mkdir(parents=True, exist_ok=True)

# ── Spatial 图：无 hires，黑色背景，矢量点 ───────────────
with plt.style.context("dark_background"):
    fig, ax = plt.subplots(figsize=(6, 6), facecolor="black")
    ax.set_facecolor("black")

    sc.pl.spatial(
        adata,
        color="leiden3",
        palette=leiden3_palette,
        img_key=None,
        spot_size=28,
        ax=ax,
        show=False,
    )

    # 保险：强制点不栅格化
    for coll in ax.collections:
        coll.set_rasterized(False)

    fig.savefig(
        OUTDIR / "leiden3_spatial_black.pdf",
        bbox_inches="tight",
        transparent=False,
        facecolor="black",
    )

    plt.show()
    plt.close(fig)

# ── UMAP 图：黑色背景，矢量点 ───────────────────────────
with plt.style.context("dark_background"):
    fig, ax = plt.subplots(figsize=(7, 6), facecolor="black")
    ax.set_facecolor("black")

    sc.pl.umap(
        adata,
        color="leiden3",
        palette=leiden3_palette,
        frameon=False,
        ax=ax,
        show=False,
    )

    # 保险：强制点不栅格化
    for coll in ax.collections:
        coll.set_rasterized(False)

    fig.savefig(
        OUTDIR / "leiden3_umap_black.pdf",
        bbox_inches="tight",
        transparent=False,
        facecolor="black",
    )

    plt.show()
    plt.close(fig)

In [ ]:


# ── 字体设置 ───────────────────────────────────────────
plt.rcParams["font.family"] = "Arial"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

# 关键：开启 Scanpy 的 vector friendly
# 点会被栅格化，PDF 更小、更不卡
sc.settings._vector_friendly = True

# ── 按 leiden3 category 顺序生成颜色 ────────────────────
adata.obs["leiden3"] = adata.obs["leiden3"].astype("category")

color_list = [
    leiden3_color_map[str(cluster)]
    for cluster in adata.obs["leiden3"].cat.categories
]
adata.uns["leiden3_colors"] = color_list

# ── 输出路径 ───────────────────────────────────────────
OUTDIR = Path(
    "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/"
    "20260427_AS_analysis/Fig2_isoform/SpatialGlue/figures0718"
)
OUTDIR.mkdir(parents=True, exist_ok=True)

# ── Spatial 图：黑色背景，光栅化点 ─────────────────────
with plt.style.context("dark_background"):
    fig, ax = plt.subplots(figsize=(6, 6), facecolor="black")
    ax.set_facecolor("black")

    sc.pl.spatial(
        adata,
        color="leiden3",
        palette=leiden3_palette,
        img_key=None,
        spot_size=28,
        ax=ax,
        show=False,
    )

    # 保险：强制点光栅化
    for coll in ax.collections:
        coll.set_rasterized(True)

    fig.savefig(
        OUTDIR / "leiden3_spatial_black_rasterized.pdf",
        bbox_inches="tight",
        transparent=False,
        facecolor="black",
        dpi=300,
    )

    plt.show()
    plt.close(fig)

# ── UMAP 图：黑色背景，光栅化点 ────────────────────────
with plt.style.context("dark_background"):
    fig, ax = plt.subplots(figsize=(7, 6), facecolor="black")
    ax.set_facecolor("black")

    sc.pl.umap(
        adata,
        color="leiden3",
        palette=leiden3_palette,
        frameon=False,
        ax=ax,
        show=False,
    )

    # 保险：强制点光栅化
    for coll in ax.collections:
        coll.set_rasterized(True)

    fig.savefig(
        OUTDIR / "leiden3_umap_black_rasterized.pdf",
        bbox_inches="tight",
        transparent=False,
        facecolor="black",
        dpi=300,
    )

    plt.show()
    plt.close(fig)

## 细分聚类

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib
import matplotlib.pyplot as plt


In [ ]:
adata=sc.read("/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260427_AS_analysis/Fig2_spatialGlue/adatas/G21-4_SG-gene-psi.h5ad")
adata

In [ ]:
color_list = [
    "#1F77B4", "#FF7F0E", "#2CA02C", "#D62728", "#9467BD", "#8C564B",
    "#E377C2", "#7F7F7F", "#BCBD22", "#17BECF",
    "#D1495B", "#5E60CE", "#FF99C8", "#A8DADC",
    "#A6CEE3", "#FDBF6F", "#B2DF8A", "#FB9A99", "#CAB2D6", "#F3CB29",
    "#A30059", "#22A884", "#FF34FF", "#809693", "#FFDBE5", "#023FA5",
    "#9CDED6",
    
    "#E76F51", "#457B9D", "#6A994E", "#C44536", "#7B2CBF", "#B56576",
    "#00B4D8", "#90BE6D", "#F9C74F", "#F9844A", "#577590", "#43AA8B",
]

In [ ]:
sc.pl.spatial(
    adata, color="leiden",palette=color_list,
    img_key=None, spot_size=28,)

In [ ]:
sc.pl.spatial(
    adata, color="leiden",palette=color_list,groups=["26"],
    img_key=None, spot_size=28,)

In [ ]:
sc.tl.leiden(adata, restrict_to=["leiden3", ["3"]], resolution=0.2, key_added='leiden4')

In [ ]:
sc.pl.spatial(
    adata, color="leiden4",palette=color_list,
    img_key=None, spot_size=28,)

In [ ]:
sc.pl.spatial(
    adata, color="leiden4",palette=color_list,groups=["2","12","26","28"],
    img_key=None, spot_size=28,)

In [ ]:
adata.obs["leiden4"] = (
    adata.obs["leiden4"]
    .astype(str)
    .replace({"12": "2", "26": "2"})
    .astype("category")
)

In [ ]:
sc.pl.spatial(
    adata, color="leiden4",palette=color_list,groups=["2","12","26","28"],
    img_key=None, spot_size=28,)

In [ ]:
sc.pl.spatial(
    adata, color="leiden4",palette=color_list,
    img_key=None, spot_size=28,)

In [ ]:
sc.pl.spatial(
    adata, color="leiden4",palette=color_list,groups=["14","0","6","16"],
    img_key=None, spot_size=28,)

In [ ]:
sc.tl.leiden(adata, restrict_to=["leiden4", ["14"]], resolution=0.2, key_added='leiden5')

In [ ]:
deconv_colors_G214 = {
    "Ext_Thal_1":     "#cac43d",  # 黄绿色
    "Oligo_1":        "#10976f",  # 深青绿
    "Ext_L23":        "#1681ec",  # 蓝色
    "Ext_L25":        "#a81e2b",  # 暗红色
    "Ext_Hpc_DG1":    "#009e73",  # 绿色
    "Oligo_2":        "#67c1dd",  # 浅蓝色
    "Ext_L56":        "#5a14a5",  # 紫色
    "Ext_Hpc_CA1":    "#f0e442",  # 明黄色
    "Ext_Thal_2":     "#b71e75",  # 洋红色
    "Ext_Pir":        "#74c476",  # 浅绿色
    "Ext_Amy_2":      "#8c510a",  # 棕色
    "Ext_L5_1":       "#fd8d3c",  # 橙色
    "Inh_4":          "#ffbe2c",  # 金黄色
    "Inh_1":          "#865123",  # 深棕色
    "Ext_Hpc_CA3":    "#56b4e9",  # 天蓝色
    "Micro":          "#9e9ac8",  # 淡紫色
    "Inh_Meis2_3":    "#6baed6",  # 浅蓝
    "Ext_L6":         "#31a354",  # 草绿色
    "Ext_L5_2":       "#e6550d",  # 橙红色
    "Astro_HYPO":     "#8ca252",  # 橄榄绿色
    "Ext_L6B":        "#3182bd",  # 中蓝色
    "Endo":           "#ec7c7f",  # 珊瑚粉
    "Astro_WM":       "#e7969c",  # 浅粉色
    "Inh_Pvalb":      "#ce6dbd",  # 粉紫色
    "Ext_ClauPyr":    "#756bb1",  # 蓝紫色
    "Nb_2":           "#2ca02c",  # 绿色
    "Inh_5":          "#7b4173",  # 深紫色
    "Astro_THAL_lat": "#bd9e39",  # 土黄色
    "Inh_6":          "#636363",  # 深灰色
    "Inh_Meis2_4":    "#9c9ede",  # 淡蓝紫
    "Ext_Med":        "#a55194",  # 紫红色

    # notebook 中没有单独指定，所以使用默认补充颜色
    "Inh_Sst":        "#aaaaaa",  # 灰色
}

In [ ]:
sc.pl.spatial(
    adata, color="leiden5",palette=color_list,
    img_key=None, spot_size=28,)

In [ ]:
sc.pl.spatial(
    adata, color="leiden5",palette=color_list,groups=["5","8"],
    img_key=None, spot_size=28,)

In [ ]:
adata.obs["leiden5"] = (
    adata.obs["leiden5"]
    .astype(str)
    .replace({"8": "5"})
    .astype("category")
)

In [ ]:
plt.rcParams['figure.figsize']=(6,6)
import seaborn as sns
sns.set(style="ticks", font_scale=1.2)

In [ ]:
({"12": "2", "26": "2"})

In [ ]:
sc.pl.spatial(
    adata, color="leiden5",palette=color_list,groups=["1","5","8","22"],
    img_key=None, spot_size=28,)

In [ ]:
sc.pl.spatial(
    adata, color="leiden5",palette=color_list,
    img_key=None, spot_size=28,)

In [ ]:
leiden5_palette = {
    "0":    "#5a14a5",
    "1":    "#ffbe2c",
    "2":    "#10976f",
    "3,0":    黑灰色1,
    "3,1":    有所区别黑灰色2,
    "4":    "#b71e75",
    "5":    "#ffbe2c",
    "6":    "#a81e2b",
    "7":    "#756bb1",
    "8":    "#ffbe2c",
    "9":    "#8c510a",
    "10":   比较暗的蓝色,"#3182bd"
    "11":   "#6baed6",
    "12":   "#10976f",
    "13":   "#9e9ac8",
    "14,0":   紫色偏橘色一点,
    "14,1":   "#ce6dbd",
    "14,2":   淡橘色,
    "15":   "#636363",
    "16":   "#fd8d3c",
    "17":   "#74c476",
    "18":   浅棕色,
    "19,0": "#f0e442",
    "19,1": "#ffbe2c",
    "20,0": "#636363",
    "20,1": "#56b4e9",
    "20,2": "#56b4e9",
    "21":   "#aaaaaa",
    "22":   "#bd9e39",
    "23":   "#aaaaaa",
    "24":   "#7b4173",
    "25,0": "#009e73",
    "25,1": "#e7969c",
    "26":   "#10976f",
    "27":   灰棕色,
    "28":   "#ec7c7f",
}

In [ ]:
leiden5_palette = {
    "0":    "#5a14a5",
    "1":    "#b8842d",   # 麦黄色
    "2":    "#10976f",
    "3,0":  "#333333",
    "3,1":  "#555555",
    "4":    "#b71e75",
    "5":    "#ffd36a",
    "6":    "#a81e2b",
    "7":    "#756bb1",
    "8":    "#ffbe2c",
    "9":    "#8c510a",
    "10":   "#1f4e79",
    "11":   "#6baed6",
    "12":   "#10976f",
    "13":   "#9e9ac8",
    "14,0": "#b45f3c",
    "14,1": "#ce6dbd",
    "14,2": "#fdae6b",
    "15":   "#746874",
    "16":   "#fd8d3c",
    "17":   "#74c476",
    "18":   "#c49a6c",
    "19,0": "#dfff00",#"#f0e442",
    "19,1": "#ffbe2c",
    "20,0": "#746874",
    "20,1": "#56b4e9",
    "20,2": "#56b4e9",
    "21":   "#19201c",
    "22":   "#bd9e39",
    "23":   "#19201c",
    "24":   "#7b4173",
    "25,0": "#009e73",
    "25,1": "#e7969c",
    "26":   "#10976f",
    "27":   "#595959",
    "28":   "#ec7c7f",
}

In [ ]:
#19201c

In [ ]:
from pathlib import Path

In [ ]:


# ── 字体设置 ───────────────────────────────────────────
plt.rcParams["font.family"] = "Arial"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

# 关键：开启 Scanpy 的 vector friendly
# 点会被栅格化，PDF 更小、更不卡
sc.settings._vector_friendly = True

# ── 输出路径 ───────────────────────────────────────────
OUTDIR = Path(
    "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/"
    "20260427_AS_analysis/Fig2_isoform/SpatialGlue/figures0718"
)
OUTDIR.mkdir(parents=True, exist_ok=True)

# ── Spatial 图：黑色背景，光栅化点 ─────────────────────
with plt.style.context("dark_background"):
    fig, ax = plt.subplots(figsize=(6, 6), facecolor="black")
    ax.set_facecolor("black")

    sc.pl.spatial(
        adata,
        color="leiden5",
        palette=leiden5_palette,
        img_key=None,
        spot_size=28,
        ax=ax,
        show=False,
    )

    # 保险：强制点光栅化
    for coll in ax.collections:
        coll.set_rasterized(True)

    fig.savefig(
        OUTDIR / "leiden3_spatial_black_rasterized.pdf",
        bbox_inches="tight",
        transparent=False,
        facecolor="black",
        dpi=300,
    )

    plt.show()
    plt.close(fig)

# ── UMAP 图：黑色背景，光栅化点 ────────────────────────
with plt.style.context("dark_background"):
    fig, ax = plt.subplots(figsize=(7, 6), facecolor="black")
    ax.set_facecolor("black")

    sc.pl.umap(
        adata,
        color="leiden5",
        palette=leiden5_palette,
        frameon=False,
        ax=ax,
        show=False,
    )

    # 保险：强制点光栅化
    for coll in ax.collections:
        coll.set_rasterized(True)

    fig.savefig(
        OUTDIR / "leiden3_umap_black_rasterized.pdf",
        bbox_inches="tight",
        transparent=False,
        facecolor="black",
        dpi=300,
    )

    plt.show()
    plt.close(fig)

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path

# ── 字体设置 ───────────────────────────────────────────
plt.rcParams["font.family"] = "Arial"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

# 关键：关闭 Scanpy 自动栅格化，保证 PDF 中点尽量保持矢量可编辑
sc.settings._vector_friendly = False

# ── 输出路径 ───────────────────────────────────────────
OUTDIR = Path(
    "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/"
    "20260427_AS_analysis/Fig2_isoform/SpatialGlue/figures0718"
)
OUTDIR.mkdir(parents=True, exist_ok=True)

# ── Spatial 图：无 hires，黑色背景，矢量点 ───────────────
with plt.style.context("dark_background"):
    fig, ax = plt.subplots(figsize=(6, 6), facecolor="black")
    ax.set_facecolor("black")

    sc.pl.spatial(
        adata,
        color="leiden5",
        palette=leiden5_palette,
        img_key=None,
        spot_size=25,
        ax=ax,
        show=False,
    )

    # 保险：强制点不栅格化
    for coll in ax.collections:
        coll.set_rasterized(False)

    fig.savefig(
        OUTDIR / "leiden3_spatial_black.pdf",
        bbox_inches="tight",
        transparent=False,
        facecolor="black",
    )

    plt.show()
    plt.close(fig)

# ── UMAP 图：黑色背景，矢量点 ───────────────────────────
with plt.style.context("dark_background"):
    fig, ax = plt.subplots(figsize=(7, 6), facecolor="black")
    ax.set_facecolor("black")

    sc.pl.umap(
        adata,
        color="leiden5",
        palette=leiden5_palette,
        frameon=False,
        ax=ax,
        show=False,
    )

    # 保险：强制点不栅格化
    for coll in ax.collections:
        coll.set_rasterized(False)

    fig.savefig(
        OUTDIR / "leiden3_umap_black.pdf",
        bbox_inches="tight",
        transparent=False,
        facecolor="black",
    )

    plt.show()
    plt.close(fig)

## 分类的热图

In [ ]:
ad1=adata.copy()

In [ ]:
ad1.var_names = ad1.var['Symbol'].astype(str)
ad1.var_names_make_unique()

In [ ]:
ad1

In [ ]:
ad1.var

In [ ]:
sc.pl.spatial(ad1,img_key=None,color=["total_counts",'n_genes_by_counts'],spot_size=25,)

In [ ]:
sc.pp.filter_cells(ad1, min_genes=50)
sc.pp.filter_cells(ad1, min_counts=10)
sc.pp.filter_genes(ad1, min_cells=5)

In [ ]:
sc.pp.normalize_total(ad1, target_sum=1e4)
sc.pp.log1p(ad1)

if "log1p" not in ad1.uns:
    ad1.uns["log1p"] = {}

ad1.uns["log1p"]["base"] = None

sc.tl.rank_genes_groups(
    ad1,
    groupby="leiden5",
    method="wilcoxon",
    n_genes=50,
    use_raw=False,
    pts=True,
)

In [ ]:
sc.tl.filter_rank_genes_groups(ad1,
                               min_in_group_fraction= 0.4,max_out_group_fraction= 0.4,
                              )

In [ ]:
# 提取每个脑区 Top marker 基因
markers_dict = {}
groups = ad1.obs['leiden5'].cat.categories.tolist()

for grp in groups:
    df = sc.get.rank_genes_groups_df(ad1, 
                                     group=grp,
                                     #key='rank_genes_groups_filtered'
                                    )
    genes = [g for g in df.head(3)['names'].tolist() if g in ad1.var_names]
    if genes:
        markers_dict[grp] = genes
markers_dict

In [ ]:
# 画图
sc.pl.matrixplot(
    ad1,
    var_names=markers_dict,
    groupby='leiden5',standard_scale="var",
    #layer='scaled',
    dendrogram=True,
    #swap_axes=True,
    # vmin=0, vmax=1,
    cmap='RdBu_r',
    #save='brain_cluster_markers.pdf'
)

In [ ]:
sc.pl.dotplot(
    ad1,
    var_names=markers_dict,
    groupby='leiden5',standard_scale="var",
    #layer='scaled',
    dendrogram=True,
    #swap_axes=True,
    # vmin=0, vmax=1,
    cmap='RdBu_r',
    save='brain_cluster_markers.pdf'
)

# 000